# ATLAS - Shipment Data Creation

## Purpose

For this notebook, the goal is to create a synthetic dataset representing time-sensitive medical shipments traveling to hospitals in New Jersey.

Real advanced therapy shipment data is private, so simulated shipment information will be created using the real hospital destinations prepared in Notebook 1.

The synthetic shipment data will later be combined with traffic-risk information in the ATLAS dashboard.

## Step 1: Import Libraries

The first step is to import the Python libraries needed for this notebook.

Pandas will be used to work with the hospital and shipment data, while NumPy will help generate simulated shipment values.

In [1]:
import pandas as pd
import numpy as np

## Step 2: Load the Cleaned Hospital Dataset

The cleaned New Jersey hospital dataset from Notebook 1 will be used to provide real hospital destinations for the simulated shipments.

This allows ATLAS to use real medical facility locations while keeping the shipment information completely synthetic.

In [3]:
hospitals_df = pd.read_csv("../data/processed/nj_hospitals_clean.csv")

hospitals_df.head()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish
0,310001,HACKENSACK UNIVERSITY MEDICAL CENTER,30 PROSPECT AVE,HACKENSACK,NJ,7601,BERGEN
1,310002,NEWARK BETH ISRAEL MEDICAL CENTER,201 LYONS AVE,NEWARK,NJ,7112,ESSEX
2,310003,PALISADES MEDICAL CENTER,7600 RIVER RD,NORTH BERGEN,NJ,7047,HUDSON
3,310005,HUNTERDON MEDICAL CENTER,2100 WESCOTT DRIVE,FLEMINGTON,NJ,8822,HUNTERDON
4,310006,ST MARY'S GENERAL HOSPITAL,350 BOULEVARD,PASSAIC,NJ,7055,PASSAIC


In [4]:
hospitals_df.shape

(78, 7)

## Step 3: Review Available Shipment Destinations

Before creating the synthetic shipments, the available New Jersey hospitals will be reviewed.

Each simulated shipment will later be assigned one of these hospitals as its destination.

In [5]:
hospitals_df[
    ["Facility ID", "Facility Name", "City/Town", "County/Parish"]
].head(10)

,Facility ID,Facility Name,City/Town,County/Parish
0,310001,HACKENSACK UNIVERSITY MEDICAL CENTER,HACKENSACK,BERGEN
1,310002,NEWARK BETH ISRAEL MEDICAL CENTER,NEWARK,ESSEX
2,310003,PALISADES MEDICAL CENTER,NORTH BERGEN,HUDSON
3,310005,HUNTERDON MEDICAL CENTER,FLEMINGTON,HUNTERDON
4,310006,ST MARY'S GENERAL HOSPITAL,PASSAIC,PASSAIC
5,310008,HOLY NAME MEDICAL CENTER,TEANECK,BERGEN
6,310009,CLARA MAASS MEDICAL CENTER,BELLEVILLE,ESSEX
7,310010,UNIVERSITY MEDICAL CENTER OF PRINCETON AT PLAI...,PLAINSBORO,MIDDLESEX
8,310011,CAPE REGIONAL MEDICAL CENTER INC,CAPE MAY COURT HOUSE,CAPE MAY
9,310012,VALLEY HOSPITAL,PARAMUS,BERGEN


## Step 4: Create Synthetic Shipments

Because real advanced therapy shipment data is not publicly available, ATLAS will use simulated shipment records.

Each shipment will contain:

- Shipment ID
- Origin
- Destination Hospital
- Destination City
- Destination County
- Scheduled Arrival Time
- Estimated Arrival Time
- Package Temperature
- Remaining Time Buffer
- Shipment Status

The hospital destinations will come from the real New Jersey hospital dataset, while the shipment information itself will be simulated.

In [7]:
np.random.seed(42)

num_shipments = 50

sampled_hospitals = hospitals_df.sample(
    n=num_shipments,
    replace=True,
    random_state=42
).reset_index(drop=True)

shipments_df = pd.DataFrame({
    "Shipment_ID": [f"AT-{1001 + i}" for i in range(num_shipments)],
    "Origin": "Raritan, NJ",
    "Destination_Hospital": sampled_hospitals["Facility Name"],
    "Destination_City": sampled_hospitals["City/Town"],
    "Destination_County": sampled_hospitals["County/Parish"]
})

shipments_df.head()

,Shipment_ID,Origin,Destination_Hospital,Destination_City,Destination_County
0,AT-1001,"Raritan, NJ",CAPITAL HEALTH REGIONAL MEDICAL CENTER,TRENTON,MERCER
1,AT-1002,"Raritan, NJ",ST JOSEPH'S UNIVERSITY MEDICAL CENTER INC,PATERSON,PASSAIC
2,AT-1003,"Raritan, NJ",NORTHBROOK BEHAVIORAL HEALTH HOSPITAL,BLACKWOOD,CAMDEN
3,AT-1004,"Raritan, NJ",THE UNIVERSITY HOSPITAL,NEWARK,ESSEX
4,AT-1005,"Raritan, NJ",NEWTON MEDICAL CENTER,NEWTON,SUSSEX


## Step 5: Add Shipment Timing

Each shipment needs a scheduled arrival time and an estimated arrival time.

The scheduled arrival represents when the shipment is expected to reach the hospital.

The estimated arrival will be simulated so some shipments arrive early, some on time, and some late.

In [9]:
base_time = pd.Timestamp("2026-10-22 08:00:00")

shipments_df["Scheduled_Arrival"] = [
    base_time + pd.Timedelta(minutes=30 * i)
    for i in range(num_shipments)
]

delay_minutes = np.random.randint(-15, 61, size=num_shipments)

shipments_df["Estimated_Arrival"] = (
    shipments_df["Scheduled_Arrival"]
    + pd.to_timedelta(delay_minutes, unit="m")
)

shipments_df[
    [
        "Shipment_ID",
        "Scheduled_Arrival",
        "Estimated_Arrival"
    ]
].head()

,Shipment_ID,Scheduled_Arrival,Estimated_Arrival
0,AT-1001,2026-10-22 08:00:00,2026-10-22 07:52:00
1,AT-1002,2026-10-22 08:30:00,2026-10-22 09:01:00
2,AT-1003,2026-10-22 09:00:00,2026-10-22 09:19:00
3,AT-1004,2026-10-22 09:30:00,2026-10-22 09:50:00
4,AT-1005,2026-10-22 10:00:00,2026-10-22 10:34:00


## Step 6: Calculate Remaining Delivery Buffer

Each shipment will be given a simulated delivery deadline 60 minutes after its scheduled arrival time.

The remaining delivery buffer shows how much time is left between the estimated arrival and that deadline.

A smaller remaining buffer means the shipment is closer to becoming at risk.

In [11]:
shipments_df["Delivery_Deadline"] = (
    shipments_df["Scheduled_Arrival"]
    + pd.Timedelta(minutes=60)
)

shipments_df["Remaining_Buffer_Min"] = (
    shipments_df["Delivery_Deadline"]
    - shipments_df["Estimated_Arrival"]
).dt.total_seconds() / 60

shipments_df[
    [
        "Shipment_ID",
        "Scheduled_Arrival",
        "Estimated_Arrival",
        "Delivery_Deadline",
        "Remaining_Buffer_Min"
    ]
].head()

,Shipment_ID,Scheduled_Arrival,Estimated_Arrival,Delivery_Deadline,Remaining_Buffer_Min
0,AT-1001,2026-10-22 08:00:00,2026-10-22 07:52:00,2026-10-22 09:00:00,68.0
1,AT-1002,2026-10-22 08:30:00,2026-10-22 09:01:00,2026-10-22 09:30:00,29.0
2,AT-1003,2026-10-22 09:00:00,2026-10-22 09:19:00,2026-10-22 10:00:00,41.0
3,AT-1004,2026-10-22 09:30:00,2026-10-22 09:50:00,2026-10-22 10:30:00,40.0
4,AT-1005,2026-10-22 10:00:00,2026-10-22 10:34:00,2026-10-22 11:00:00,26.0


## Step 7: Assign Shipment Risk Levels

The remaining delivery buffer will be used to assign each shipment a simple risk level.

For this proof of concept:

- Low Risk: more than 60 minutes remaining
- Medium Risk: 30 to 60 minutes remaining
- High Risk: less than 30 minutes remaining

These thresholds are simulated and are only being used to demonstrate how ATLAS could prioritize shipments.

In [13]:
def assign_risk(buffer):
    if buffer > 60:
        return "Low Risk"
    elif buffer >= 30:
        return "Medium Risk"
    else:
        return "High Risk"

shipments_df["Shipment_Status"] = shipments_df[
    "Remaining_Buffer_Min"
].apply(assign_risk)

shipments_df[
    [
        "Shipment_ID",
        "Remaining_Buffer_Min",
        "Shipment_Status"
    ]
].head(10)

,Shipment_ID,Remaining_Buffer_Min,Shipment_Status
0,AT-1001,68.0,Low Risk
1,AT-1002,29.0,High Risk
2,AT-1003,41.0,Medium Risk
3,AT-1004,40.0,Medium Risk
4,AT-1005,26.0,High Risk
5,AT-1006,72.0,Low Risk
6,AT-1007,74.0,Low Risk
7,AT-1008,70.0,Low Risk
8,AT-1009,22.0,High Risk
9,AT-1010,72.0,Low Risk


## Step 8: Add Simulated Package Temperature

Some advanced therapy shipments may require temperature-controlled handling.

For this proof of concept, each shipment will be assigned a simulated package temperature so ATLAS can demonstrate how temperature information could be monitored alongside delivery timing.

These values are synthetic and are only used for demonstration.

In [14]:
shipments_df["Temperature_C"] = np.round(
    np.random.uniform(2.0, 8.0, size=num_shipments),
    1
)

shipments_df[
    [
        "Shipment_ID",
        "Temperature_C",
        "Shipment_Status"
    ]
].head(10)

,Shipment_ID,Temperature_C,Shipment_Status
0,AT-1001,2.6,Low Risk
1,AT-1002,4.2,High Risk
2,AT-1003,6.0,Medium Risk
3,AT-1004,6.0,Medium Risk
4,AT-1005,5.5,High Risk
5,AT-1006,3.6,Low Risk
6,AT-1007,5.4,Low Risk
7,AT-1008,4.3,Low Risk
8,AT-1009,7.8,High Risk
9,AT-1010,7.1,Low Risk


## Step 9: Review the Synthetic Shipment Dataset

Before saving the dataset, the final shipment records will be reviewed to confirm that all expected fields were created correctly.

In [15]:
shipments_df.head()

,Shipment_ID,Origin,Destination_Hospital,Destination_City,Destination_County,Scheduled_Arrival,Estimated_Arrival,Delivery_Deadline,Remaining_Buffer_Min,Shipment_Status,Temperature_C
0,AT-1001,"Raritan, NJ",CAPITAL HEALTH REGIONAL MEDICAL CENTER,TRENTON,MERCER,2026-10-22 08:00:00,2026-10-22 07:52:00,2026-10-22 09:00:00,68.0,Low Risk,2.6
1,AT-1002,"Raritan, NJ",ST JOSEPH'S UNIVERSITY MEDICAL CENTER INC,PATERSON,PASSAIC,2026-10-22 08:30:00,2026-10-22 09:01:00,2026-10-22 09:30:00,29.0,High Risk,4.2
2,AT-1003,"Raritan, NJ",NORTHBROOK BEHAVIORAL HEALTH HOSPITAL,BLACKWOOD,CAMDEN,2026-10-22 09:00:00,2026-10-22 09:19:00,2026-10-22 10:00:00,41.0,Medium Risk,6.0
3,AT-1004,"Raritan, NJ",THE UNIVERSITY HOSPITAL,NEWARK,ESSEX,2026-10-22 09:30:00,2026-10-22 09:50:00,2026-10-22 10:30:00,40.0,Medium Risk,6.0
4,AT-1005,"Raritan, NJ",NEWTON MEDICAL CENTER,NEWTON,SUSSEX,2026-10-22 10:00:00,2026-10-22 10:34:00,2026-10-22 11:00:00,26.0,High Risk,5.5


In [16]:
shipments_df.shape

(50, 11)

In [17]:
shipments_df["Shipment_Status"].value_counts()

Shipment_Status
High Risk      17
Medium Risk    17
Low Risk       16
Name: count, dtype: int64

## Step 10: Save the Synthetic Shipment Dataset

The completed synthetic shipment dataset will now be saved in the `data/processed` folder.

This dataset will later be combined with the hospital and traffic datasets during the ATLAS analysis.

In [18]:
shipments_df.to_csv(
    "../data/processed/synthetic_shipments.csv",
    index=False
)